# Raining Hoops — WNBA 2026 NBA API Ingestion

This notebook runs the Raining Hoops pipeline in Google Colab using the same `nba_api` PlayerGameLogs endpoint. The endpoint supports LeagueID, Season, SeasonType, and Month; WNBA regular-season months are limited to May–September.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/samyak0407/raininghoops.git"
REPO_DIR = "/content/raininghoops"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
%cd /content/raininghoops
print("Repository ready:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt
print("Dependencies installed.")

In [ ]:
import nba_api
from nba_api.stats.endpoints import playergamelogs
print("nba_api version:", getattr(nba_api, "__version__", "installed"))
print("PlayerGameLogs imported successfully:", playergamelogs.PlayerGameLogs.__name__)

In [ ]:
import requests
from src.data.nba_stats_client import DEFAULT_HEADERS
params = {"LeagueID":"10","Season":"2026","SeasonType":"Regular Season","Month":"5"}
url = "https://stats.nba.com/stats/playergamelogs"
try:
    response = requests.get(url, params=params, headers=DEFAULT_HEADERS, timeout=20)
    print("HTTP status:", response.status_code)
    print("Response bytes:", len(response.content))
    if response.status_code != 200:
        raise RuntimeError(f"NBA Stats returned HTTP {response.status_code}.")
    print("PRECHECK PASSED — Colab can reach the NBA Stats endpoint.")
except requests.RequestException as exc:
    print("PRECHECK FAILED — Colab could not reach stats.nba.com.")
    print("Network error:", exc)
    raise SystemExit("Stop here; do not run the full ingestion until the endpoint is reachable from Colab.")

In [ ]:
from src.data.nba_stats_client import OfficialStatsClient, WNBA
from src.data.normalize import normalize_player_game_logs
from src.data.quality import assert_valid_player_game_logs, validate_player_game_logs
client = OfficialStatsClient(timeout=60, max_retries=2)
raw = client.player_game_logs(season="2026", season_type="Regular Season", league=WNBA)
print("Raw rows:", f"{len(raw):,}")
normalized = normalize_player_game_logs(raw)
assert_valid_player_game_logs(normalized)
quality = validate_player_game_logs(normalized)
print("Normalized rows:", f"{len(normalized):,}")
print("Players:", f"{normalized['player_id'].nunique():,}")
print("Games:", f"{normalized['game_id'].nunique():,}")
print("Date range:", normalized['game_date'].min(), "->", normalized['game_date'].max())
display(normalized.head())
print("Quality report:")
print(quality)

In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone
output_dir = Path("data/raw/player_game_logs")
output_dir.mkdir(parents=True, exist_ok=True)
output = output_dir / "wnba_2026_regular_season.parquet"
normalized.to_parquet(output, index=False)
manifest = {
    "league":"WNBA","season":"2026","season_type":"Regular Season",
    "source":"NBA.com Stats via nba_api",
    "retrieved_at_utc":datetime.now(timezone.utc).isoformat(),
    "rows":len(normalized),"players":int(normalized["player_id"].nunique()),
    "games":int(normalized["game_id"].nunique()),
    "date_min":str(normalized["game_date"].min()),
    "date_max":str(normalized["game_date"].max()),
    "quality":quality,"output":str(output)
}
manifest_path = output.with_suffix(".json")
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
print("Saved:", output)
print("Manifest:", manifest_path)

## Next step

Once the verified WNBA 2026 dataset exists, the next layer is rolling opportunity, role shifts, teammate availability, matchup context, prediction, and evaluation. StatPick remains a separate market-intelligence input.